In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_downpayment_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "DownPaymentId	string,"
        # "SourceSystem	string,"
        # "PortfolioKey	int,"
        # "PortfolioName	string,"
        # "IngestionTimestamp	timestamp,"
        # "BatchId	    string,"
        # "DownPaymentKey	string,"
        # "AgreementKey	string,"
        # "ApplicationKey string,"
        # "ApplicationGRP string,"
        # "BGMProgramStep int,"
        # "BGMProgramName string,"
        # "TransactionDate date,"
        # "ExpiryDate     date,"
        # "StartDate      date,"
        # "ToleranceDays  decimal(10,0),"
        # "ActivationAmount decimal(15,2),"
        # "ToleranceAmount  decimal(15,2),"
        # "ActualCreditedAmount decimal(15,2),"
        # "ActivationStatusCode string,"
        # "ActivationStatus string,"
        # "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Downpayment"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "profits",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== arch_inp_bgm_schedule ==================
        arch_inp_bgm_schedule_extra_filters = None

        if arch_inp_bgm_schedule_extra_filters is not None:
            arch_inp_bgm_schedule_extra_filters.append(date_filter)
            arch_inp_bgm_schedule_final_filters = " AND ".join(
                arch_inp_bgm_schedule_extra_filters
            )
        else:
            arch_inp_bgm_schedule_final_filters = date_filter

        arch_inp_bgm_schedule_selected_fields = [
            "char_03",
            "ext_application_id",
            "insert_tmstamp",
            "bgm_program_step",
            "ext_application_grp_id",
            "bgm_program",
            "bgm_trx_date",
            "date_09",
            "date_10",
            "number_10",
            "amount_09",
            "amount_10",
            "amount_08",
            "process_status",
            "businessdate",
        ]

        arch_inp_bgm_schedule = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_arch_inp_bgm_schedule",
            arch_inp_bgm_schedule_final_filters,
            arch_inp_bgm_schedule_selected_fields,
        )

        arch_inp_bgm_schedule_primary_key_fields_1 = [
            arch_inp_bgm_schedule.char_03,
            arch_inp_bgm_schedule.ext_application_id,
            arch_inp_bgm_schedule.insert_tmstamp,
            arch_inp_bgm_schedule.bgm_program_step,
            lit("profits"),
        ]

        arch_inp_bgm_schedule_primary_key_fields_2 = [
            arch_inp_bgm_schedule.char_03,
            arch_inp_bgm_schedule.ext_application_id,
            arch_inp_bgm_schedule.insert_tmstamp,
            arch_inp_bgm_schedule.bgm_program_step,
        ]

        arch_inp_bgm_schedule = (
            arch_inp_bgm_schedule.withColumn(
                "DownPaymentId",
                sha2(concat_ws(";", *arch_inp_bgm_schedule_primary_key_fields_1), 256),
            )
            .withColumn(
                "DownPaymentKey",
                concat_ws(";", *arch_inp_bgm_schedule_primary_key_fields_2),
            )
            .where(arch_inp_bgm_schedule.bgm_program_step == 0)
        )

        arch_inp_bgm_schedule_portfolio_join_condition = (
            substring(arch_inp_bgm_schedule["char_03"], 1, 3) == portfolio["Prefix"]
        )

        target_table_profits = arch_inp_bgm_schedule.join(
            portfolio,
            arch_inp_bgm_schedule_portfolio_join_condition,
            "left",
        )
        target_table_profits = target_table_profits.withColumn(
            "ActivationStatus",
            when(arch_inp_bgm_schedule.process_status == "0", "Pending")
            .when(arch_inp_bgm_schedule.process_status == "1", "Activated")
            .when(
                arch_inp_bgm_schedule.process_status == "3",
                "Rejected - Not enough downpayment",
            )
            .when(arch_inp_bgm_schedule.process_status == "4", "Cancelled by user")
            .when(arch_inp_bgm_schedule.process_status == "5", "Cancelled by FLDT")
            .when(
                arch_inp_bgm_schedule.process_status == "6",
                "Cancelled - No Application Sent By FLDT",
            )
            .otherwise(None),
        )
        # ================== Selection of Final Columns ==================

        result_df = target_table_profits.select(
            arch_inp_bgm_schedule.DownPaymentId.cast("string"),
            lit("profits").cast("String").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("int").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            arch_inp_bgm_schedule.DownPaymentKey.cast("string"),
            arch_inp_bgm_schedule.char_03.alias("AgreementKey"),
            arch_inp_bgm_schedule.ext_application_id.alias("ApplicationKey"),
            arch_inp_bgm_schedule.ext_application_grp_id.alias("ApplicationGRP"),
            arch_inp_bgm_schedule.bgm_program_step.alias("BGMProgramStep"),
            arch_inp_bgm_schedule.bgm_program.alias("BGMProgramName"),
            arch_inp_bgm_schedule.bgm_trx_date.alias("TransactionDate"),
            arch_inp_bgm_schedule.date_09.alias("ExpiryDate"),
            arch_inp_bgm_schedule.date_10.alias("StartDate"),
            arch_inp_bgm_schedule.number_10.alias("ToleranceDays"),
            arch_inp_bgm_schedule.amount_09.alias("ActivationAmount"),
            arch_inp_bgm_schedule.amount_10.alias("ToleranceAmount"),
            arch_inp_bgm_schedule.amount_08.alias("ActualCreditedAmount"),
            arch_inp_bgm_schedule.process_status.alias("ActivationStatusCode"),
            target_table_profits.ActivationStatus.cast("string").alias(
                "ActivationStatus"
            ),
            arch_inp_bgm_schedule.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_downpayment_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)